This notebook examines two questions that sit at the heart of retail basket strategy and customer lifecycle management: **what do customers buy together** and
    **how many customers return as their order count grows**

**Which department pairs are most frequently bought together?**

In [3]:
department_co_occurrence = spark.sql("""
WITH order_depts AS (
SELECT 
    DISTINCT op.order_id, 
    d.department
FROM order_products op
JOIN products p ON op.product_id = p.product_id
JOIN departments d ON p.department_id = d.department_id
),
dept_pairs AS (
SELECT
    a.department AS dept_a, 
    b.department AS dept_b,
    COUNT(DISTINCT a.order_id) AS co_occurrence
FROM order_depts a
JOIN order_depts b ON a.order_id = b.order_id
AND a.department < b.department
GROUP BY a.department, b.department
)
SELECT
    dp.dept_a, 
    dp.dept_b, 
    dp.co_occurrence 
FROM dept_pairs dp
ORDER BY dp.co_occurrence DESC
""").toPandas()

department_co_occurrence.head(10).style.format(
    {'co_occurrence': '{:,}'}).hide(axis='index')

StatementMeta(, 95fb5c02-40d0-42f0-98b7-a429eb7793bd, 6, Finished, Available, Finished, False)

dept_a,dept_b,co_occurrence
dairy eggs,produce,"1,841,091"
produce,snacks,"1,128,643"
beverages,produce,"1,117,258"
dairy eggs,snacks,"1,079,502"
beverages,dairy eggs,"1,074,107"
frozen,produce,"999,677"
pantry,produce,"961,552"
dairy eggs,frozen,"950,433"
dairy eggs,pantry,"902,418"
beverages,snacks,"767,732"


_The dairy eggs + produce pair appears in 1,841,091 orders by far the most common basket combination, more than 60% higher than the second-ranked pair Produce appears in 7 of the top 10 pairs follow by dairy eggs appears in 6._

**Which department has the strongest affinity pull on others?** \
Raw co-occurrence counts are heavily influenced by department size - larger departments naturally appear in more baskets. To measure the true relationship between departments, we calculate the affinity percentage by dividing the co-occurrence count by the total orders containing Department A.

A higher affinity percentage indicates that customers who purchase from Department A are also likely to purchase from Department B in the same order. This insight can help identify opportunities for cross-selling, product bundling, and targeted promotions.

In [4]:
department_affinity = spark.sql("""
WITH order_depts AS (
SELECT 
    DISTINCT op.order_id, 
    d.department
FROM order_products op
JOIN products p ON op.product_id = p.product_id
JOIN departments d ON p.department_id = d.department_id
),
dept_pairs AS (
SELECT
    a.department AS dept_a, 
    b.department AS dept_b,
    COUNT(DISTINCT a.order_id) AS co_occurrence
FROM order_depts a
JOIN order_depts b ON a.order_id = b.order_id
AND a.department < b.department
GROUP BY a.department, b.department
),
dept_totals AS (
SELECT 
    department,
    COUNT(DISTINCT order_id) AS dept_orders
FROM order_depts
GROUP BY department
)
SELECT
    dp.dept_a, 
    dp.dept_b, 
    dp.co_occurrence, 
    dt.dept_orders AS dept_a_total_orders,
    ROUND(dp.co_occurrence * 100.0 / dt.dept_orders, 1) AS affinity_pct
FROM dept_pairs dp
JOIN dept_totals dt ON dp.dept_a = dt.department
WHERE dp.co_occurrence > 100000
ORDER BY affinity_pct DESC
""").toPandas()

display(department_affinity.head(5).style.format({'co_occurrence': '{:,}', 
'dept_a_total_orders': '{:,}', 'affinity_pct': '{:.1f}%'}).hide(axis='index'))

display(department_affinity.tail(5).style.format({'co_occurrence': '{:,}', 
'dept_a_total_orders': '{:,}', 'affinity_pct': '{:.1f}%'}).hide(axis='index'))

StatementMeta(, 95fb5c02-40d0-42f0-98b7-a429eb7793bd, 7, Finished, Available, Finished, False)

dept_a,dept_b,co_occurrence,dept_a_total_orders,affinity_pct
canned goods,produce,"624,650","710,721",87.9%
meat seafood,produce,"524,335","599,017",87.5%
dry goods pasta,produce,"542,461","623,738",87.0%
international,produce,"201,148","231,253",87.0%
deli,produce,"690,186","802,581",86.0%


dept_a,dept_b,co_occurrence,dept_a_total_orders,affinity_pct
bakery,personal care,"107,883","917,980",11.8%
dairy eggs,personal care,"234,055","2,264,738",10.3%
frozen,international,"116,190","1,232,089",9.4%
dairy eggs,international,"183,622","2,264,738",8.1%
beverages,international,"116,044","1,518,833",7.6%


_Produce has the strongest cross-department affinity, appearing in over 85% of orders containing canned goods, meat & seafood, deli and other major departments. This suggests produce is a key basket anchor and presents strong opportunities for cross-selling and bundled promotions._

_These lower-affinity combinations suggest that customers typically purchase these departments independently, making them less suitable for natural bundle promotions._

**Which Products has the strongest affinity pull on others?**

In [5]:
product_affinity = spark.sql("""
WITH order_products_cte AS (
SELECT DISTINCT
    op.order_id,
    p.product_name
FROM order_products op
JOIN products p ON op.product_id = p.product_id
),
product_pairs AS (
SELECT
    a.product_name AS product_a,
    b.product_name AS product_b,
    COUNT(DISTINCT a.order_id) AS co_occurrence
FROM order_products_cte a
JOIN order_products_cte b ON a.order_id = b.order_id
AND a.product_name < b.product_name
GROUP BY a.product_name, b.product_name
),
product_totals AS (
SELECT
    product_name,
    COUNT(DISTINCT order_id) AS product_total_orders
FROM order_products_cte
GROUP BY product_name
)
SELECT
    pp.product_a,
    pp.product_b,
    pp.co_occurrence,
    pt.product_total_orders,
    ROUND(pp.co_occurrence * 100.0 / pt.product_total_orders, 1) AS affinity_pct
FROM product_pairs pp
JOIN product_totals pt ON pp.product_a = pt.product_name
WHERE pp.co_occurrence > 10000
ORDER BY affinity_pct DESC
""").toPandas()

display(product_affinity.head(5).style.format({'co_occurrence': '{:,}',
'product_total_orders': '{:,}','affinity_pct': '{:.1f}%'}).hide(axis='index'))

display(product_affinity.tail(5).style.format({'co_occurrence': '{:,}',
'product_total_orders': '{:,}','affinity_pct': '{:.1f}%'}).hide(axis='index'))

StatementMeta(, 95fb5c02-40d0-42f0-98b7-a429eb7793bd, 8, Finished, Available, Finished, False)

product_a,product_b,co_occurrence,product_total_orders,affinity_pct
Lime Sparkling Water,Sparkling Water Grapefruit,"13,789","48,512",28.4%
Apple Honeycrisp Organic,Bag of Organic Bananas,"24,383","87,272",27.9%
Bunched Cilantro,Limes,"13,049","47,450",27.5%
Jalapeno Peppers,Limes,"11,940","44,460",26.9%
Organic Kiwi,Organic Strawberries,"13,003","52,021",25.0%


product_a,product_b,co_occurrence,product_total_orders,affinity_pct
Banana,Organic Kiwi,"10,951","491,291",2.2%
Banana,Organic Small Bunch Celery,"11,036","491,291",2.2%
Banana,Lime Sparkling Water,"10,170","491,291",2.1%
Banana,Unsweetened Original Almond Breeze Almond Milk,"10,072","491,291",2.1%
Banana,Raspberries,"10,418","491,291",2.1%


_Products with the highest affinity are not necessarily the most frequently purchased items. For example, Lime Sparkling Water and Sparkling Water Grapefruit have the highest affinity (28.4%), meaning that nearly 3 out of 10 orders containing Lime Sparkling Water also include Sparkling Water Grapefruit. Similarly, Apple Honeycrisp Organic is frequently paired with Bag of Organic Bananas (27.9%), while Bunched Cilantro and Limes (27.5%) and Jalapeño Peppers and Limes (26.9%) show strong complementary purchasing behavior._

_In contrast, although Bananas are the most purchased product (491,291 total orders), their affinity with any single product is relatively low (around 2.1–2.2%). This suggests that bananas are a staple item purchased alongside a wide variety of products rather than having a strong association with one specific item._

#### Cohort Analysis

Track how customer retention changes across successive purchases to identify where the largest drop-off occurs.

**What percentage of customers return to place a 2nd, 3rd, 4th and 5th.... order?**

In [6]:
customer_retention = spark.sql("""
WITH cohort_assigned AS (
SELECT
    user_id,
    order_number
FROM orders
),
cohort_size AS (
SELECT
    COUNT(DISTINCT user_id) AS total_customers
FROM orders
)
SELECT
    order_number,
    COUNT(DISTINCT user_id) AS active_users,
    total_customers,
    ROUND(COUNT(DISTINCT user_id) * 100.0 / total_customers, 1) AS retention_pct
FROM cohort_assigned
CROSS JOIN cohort_size
GROUP BY order_number, total_customers
ORDER BY order_number
""").toPandas()

display(customer_retention.head(5).style.format({'active_users': '{:,}','total_customers': '{:,}',
'retention_pct': '{:.1f}%'}).hide(axis='index'))

display(customer_retention.tail(5).style.format({'active_users': '{:,}','total_customers': '{:,}',
'retention_pct': '{:.1f}%'}).hide(axis='index'))

StatementMeta(, 95fb5c02-40d0-42f0-98b7-a429eb7793bd, 9, Finished, Available, Finished, False)

order_number,active_users,total_customers,retention_pct
1,"206,209","206,209",100.0%
2,"206,209","206,209",100.0%
3,"206,209","206,209",100.0%
4,"206,209","206,209",100.0%
5,"182,223","206,209",88.4%


order_number,active_users,total_customers,retention_pct
96,"1,592","206,209",0.8%
97,"1,525","206,209",0.7%
98,"1,471","206,209",0.7%
99,"1,421","206,209",0.7%
100,"1,374","206,209",0.7%


**How does customer retention change across successive order frequency brackets?** \
Customers are grouped into order brackets to simplify the retention journey.

In [7]:
customer_retention_brackets = spark.sql("""
WITH cohort_assigned AS (
SELECT
    user_id,
    order_number
FROM orders
),
cohort_size AS (
SELECT
    COUNT(DISTINCT user_id) AS total_customers
FROM orders
)
SELECT
    CASE
        WHEN order_number BETWEEN 1 AND 5 THEN '1-5'
        WHEN order_number BETWEEN 6 AND 10 THEN '6-10'
        WHEN order_number BETWEEN 11 AND 20 THEN '11-20'
        WHEN order_number BETWEEN 21 AND 40 THEN '21-40'
        WHEN order_number BETWEEN 41 AND 60 THEN '41-60'
        ELSE '61-100'
    END AS order_bracket,
    COUNT(DISTINCT user_id) AS active_users,
    total_customers,
    ROUND(COUNT(DISTINCT user_id) * 100.0 / total_customers, 1) AS retention_pct
FROM cohort_assigned
CROSS JOIN cohort_size
GROUP BY order_bracket, total_customers
ORDER BY retention_pct DESC
""").toPandas()

customer_retention_brackets.style.format({'active_users': '{:,}','total_customers': '{:,}',
'retention_pct': '{:.1f}%'}).hide(axis='index')

StatementMeta(, 95fb5c02-40d0-42f0-98b7-a429eb7793bd, 10, Finished, Available, Finished, False)

order_bracket,active_users,total_customers,retention_pct
1-5,"206,209","206,209",100.0%
6-10,"162,633","206,209",78.9%
11-20,"101,696","206,209",49.3%
21-40,"50,731","206,209",24.6%
41-60,"17,902","206,209",8.7%
61-100,"6,589","206,209",3.2%


_**Early retention (1–10 orders):** Customer retention remains relatively strong through the first ten orders. This suggests that once customers establish an early purchasing habit, many continue ordering._

_**Mid-term attrition (11–40 orders):** Retention declines more noticeably over the next purchasing stages, falling to 49.3% for customers reaching 11–20 orders and 24.6% by 21–40 orders. This represents the largest reduction in the active customer base._

_**Long-term loyalty (41–100 orders):** Beyond 40 orders, the customer base becomes much smaller but increasingly loyal, representing a valuable core of highly engaged repeat customers._

_Strategies that encourage customers to progress beyond their first 10–20 orders are likely to have the greatest impact on long-term customer lifetime value, while the small group of high-frequency customers represents a loyal segment that should be retained through personalised offers and loyalty programmes._

**Where do customers drop off most frequently in their purchasing journey?**

In [8]:
order_dropoff = spark.sql("""
WITH order_counts AS (
SELECT
    order_number,
    COUNT(DISTINCT user_id) AS users_at_order
FROM orders
GROUP BY order_number
),
order_stats AS (
SELECT
    order_number,
    users_at_order,
    LAG(users_at_order) OVER (ORDER BY order_number) AS prev_order_users
FROM order_counts
)
SELECT
    order_number,
    users_at_order,
    prev_order_users,
    users_at_order - prev_order_users AS user_drop,
    ROUND((prev_order_users - users_at_order) * 100.0 / prev_order_users, 1
    ) AS dropoff_pct
FROM order_stats
ORDER BY order_number
""").toPandas()

order_dropoff.fillna(0).style.format({'users_at_order': '{:,}',
'prev_order_users': '{:,}','user_drop': '{:,}',
'dropoff_pct': '{:.1f}%'}).hide(axis='index')

StatementMeta(, 95fb5c02-40d0-42f0-98b7-a429eb7793bd, 11, Finished, Available, Finished, False)

order_number,users_at_order,prev_order_users,user_drop,dropoff_pct
1,"206,209",0.0,0.0,0.0%
2,"206,209","206,209.0",0.0,0.0%
3,"206,209","206,209.0",0.0,0.0%
4,"206,209","206,209.0",0.0,0.0%
5,"182,223","206,209.0","-23,986.0",11.6%
6,"162,633","182,223.0","-19,590.0",10.8%
7,"146,468","162,633.0","-16,165.0",9.9%
8,"132,618","146,468.0","-13,850.0",9.5%
9,"120,918","132,618.0","-11,700.0",8.8%
10,"110,728","120,918.0","-10,190.0",8.4%


_The largest customer losses occur immediately after the fourth order, where 23,986 customers (11.6%) fail to progress to a fifth purchase. Although customers continue to drop off with each subsequent order_

_Overall, the results suggest that increasing early customer retention and leveraging high-affinity product relationships offer the greatest opportunities to improve customer lifetime value and basket size_